[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_12/NB_bishop_ch12_tft.ipynb)

# Temporal Fusion Transformer for Financial Forecasting

**Bishop & Bishop (2024), Chapter 12 — Transformers**

This notebook demonstrates the **core ideas behind the Temporal Fusion Transformer (TFT)** applied to financial time series forecasting, using only NumPy and scikit-learn.

We implement a simplified temporal attention mechanism that learns to weight lagged features (past returns, volatility, volume) when predicting future returns — illustrating how the self-attention mechanism from Chapter 12 gives transformers an advantage over simple sequential models.

**Key concepts:**
- Scaled dot-product attention applied to temporal features
- Variable selection via learned attention weights
- Comparison with equal-weighted (linear) baselines

**Course:** Neural Networks and Deep Learning with Business Applications

**Author:** Daniel Traian PELE — Bucharest University of Economic Studies

In [ ]:
# Run this cell in Google Colab to install dependencies
!pip install -q yfinance


In [ ]:
# ── Setup ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

# Course colour palette
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# Global rcParams
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)
print('Setup complete.')

## 1. Load S&P 500 Data

We download daily S&P 500 data (2015--2024) and compute log returns, which will serve as our prediction target.

In [ ]:
import yfinance as yf

# Download S&P 500 daily data
sp500 = yf.download('^GSPC', start='2015-01-01', end='2024-01-01', progress=False)
sp500 = sp500[['Close', 'Volume']].copy()
sp500.columns = ['Close', 'Volume']

# Compute log returns
sp500['Return'] = np.log(sp500['Close'] / sp500['Close'].shift(1))
sp500.dropna(inplace=True)

print(f'Date range : {sp500.index[0].date()} to {sp500.index[-1].date()}')
print(f'Observations: {len(sp500):,}')
print(f'Mean return : {sp500["Return"].mean():.6f}')
print(f'Std return  : {sp500["Return"].std():.6f}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(sp500.index, sp500['Close'], color=COLORS['blue'], lw=1)
ax1.set_ylabel('Close Price')
ax1.set_title('S&P 500 Daily Close (2015--2024)', fontweight='bold')

ax2.plot(sp500.index, sp500['Return'], color=COLORS['red'], lw=0.5, alpha=0.7)
ax2.set_ylabel('Log Return')
ax2.set_xlabel('Date')
ax2.set_title('Daily Log Returns', fontweight='bold')

fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_sp500')
plt.show()

## 2. Feature Engineering

We create a feature matrix inspired by the TFT's multi-horizon input structure:

| Feature | Description |
|---------|-------------|
| $r_{t-1}, \ldots, r_{t-5}$ | Lagged daily log returns |
| $\sigma_{20,t}$ | 20-day rolling standard deviation (realised volatility) |
| $\Delta V_t$ | Percentage change in trading volume |

The target is the next-day return $r_{t+1}$.

In [ ]:
# ── Build feature matrix ─────────────────────────────────────────
df = sp500.copy()

# Lagged returns: r_{t-1} ... r_{t-5}
for lag in range(1, 6):
    df[f'r_lag{lag}'] = df['Return'].shift(lag)

# Rolling 20-day realised volatility
df['vol_20d'] = df['Return'].rolling(20).std()

# Volume percentage change
df['vol_chg'] = df['Volume'].pct_change()

# Target: next-day return
df['target'] = df['Return'].shift(-1)

# Drop rows with NaN
df.dropna(inplace=True)

feature_cols = [f'r_lag{i}' for i in range(1, 6)] + ['vol_20d', 'vol_chg']
print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'Samples after feature engineering: {len(df):,}')
print(f'\nFeature statistics:')
df[feature_cols].describe().round(6)

In [ ]:
# ── Train / Test split (80/20, chronological) ───────────────────
split_idx = int(len(df) * 0.8)

X_train = df[feature_cols].values[:split_idx]
y_train = df['target'].values[:split_idx]
X_test  = df[feature_cols].values[split_idx:]
y_test  = df['target'].values[split_idx:]
dates_test = df.index[split_idx:]

# Standardise using training statistics
mu_X, sigma_X = X_train.mean(axis=0), X_train.std(axis=0)
X_train_s = (X_train - mu_X) / sigma_X
X_test_s  = (X_test  - mu_X) / sigma_X

print(f'Training samples : {len(X_train):,}  ({df.index[0].date()} to {df.index[split_idx-1].date()})')
print(f'Test samples     : {len(X_test):,}   ({df.index[split_idx].date()} to {df.index[-1].date()})')

## 3. Simple Temporal Attention Mechanism (NumPy)

The core insight from the Temporal Fusion Transformer is that **not all past time steps are equally informative**. The TFT uses a multi-head attention mechanism to learn which historical inputs matter most.

Here we implement a simplified version in pure NumPy. For each sample, we treat the 7 features as a sequence of "temporal tokens" and compute **scaled dot-product attention** (Bishop Ch 12, Eq. 12.5):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

We learn query/key/value projection vectors via gradient descent to minimise MSE on the training set.

In [ ]:
def softmax(z):
    """Numerically stable softmax over last axis."""
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


class TemporalAttentionModel:
    """
    Simplified temporal attention for tabular time-series features.

    For each sample x (shape d,):
      - Compute query q = W_q @ x, key k_i = W_k @ x, value v_i = x_i
      - Attention weights alpha_i = softmax(q * k_i / sqrt(d))
      - Output = sum(alpha_i * v_i) fed through a linear output layer
    """

    def __init__(self, d, lr=1e-3):
        self.d = d
        self.lr = lr
        # Learnable parameters (small random init)
        self.w_q = np.random.randn(d) * 0.01   # query vector
        self.w_k = np.random.randn(d) * 0.01   # key vector
        self.w_out = np.random.randn(d) * 0.01 # output projection
        self.bias = 0.0

    def _attention_weights(self, X):
        """Compute attention weights for each sample. X: (n, d)."""
        q = X @ self.w_q                          # (n,)
        k = X @ self.w_k                          # (n,)
        # Score each feature: broadcast q,k over features
        scores = np.outer(q, np.ones(self.d)) * X * self.w_k[None, :]  # (n, d)
        scores = scores / np.sqrt(self.d)
        return softmax(scores)                     # (n, d)

    def predict(self, X):
        """Predict y = attention-weighted combination of features."""
        alpha = self._attention_weights(X)         # (n, d)
        context = np.sum(alpha * X, axis=1)        # (n,)  weighted sum
        return context * self.w_out.sum() + self.bias, alpha

    def fit(self, X, y, epochs=500, verbose=True):
        """Train with simple gradient descent on MSE loss."""
        losses = []
        n = len(y)
        for epoch in range(epochs):
            y_pred, alpha = self.predict(X)
            residual = y_pred - y
            loss = np.mean(residual ** 2)
            losses.append(loss)

            # Numerical gradient (parameter-by-parameter)
            eps = 1e-5
            for param_name in ['w_q', 'w_k', 'w_out']:
                param = getattr(self, param_name)
                grad = np.zeros_like(param)
                for i in range(len(param)):
                    param[i] += eps
                    loss_plus = np.mean((self.predict(X)[0] - y) ** 2)
                    param[i] -= 2 * eps
                    loss_minus = np.mean((self.predict(X)[0] - y) ** 2)
                    param[i] += eps  # restore
                    grad[i] = (loss_plus - loss_minus) / (2 * eps)
                setattr(self, param_name, param - self.lr * grad)

            # Bias gradient
            self.bias -= self.lr * 2 * np.mean(residual)

            if verbose and (epoch + 1) % 100 == 0:
                print(f'  Epoch {epoch+1:4d}  MSE = {loss:.8f}')

        return losses


# ── Train the attention model ────────────────────────────────────
np.random.seed(42)
attn_model = TemporalAttentionModel(d=len(feature_cols), lr=5e-4)
print('Training temporal attention model...')
attn_losses = attn_model.fit(X_train_s, y_train, epochs=500, verbose=True)
print('Done.')

In [ ]:
# ── Visualise learned attention weights ──────────────────────────
_, alpha_train = attn_model.predict(X_train_s)
mean_alpha = alpha_train.mean(axis=0)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(feature_cols, mean_alpha, color=COLORS['blue'], edgecolor='k', linewidth=0.6)

# Highlight the most attended feature
max_idx = np.argmax(mean_alpha)
bars[max_idx].set_color(COLORS['red'])

ax.set_ylabel('Mean Attention Weight')
ax.set_xlabel('Feature')
ax.set_title('Learned Temporal Attention Weights (Training Set)', fontweight='bold')
ax.axhline(1.0 / len(feature_cols), ls='--', color=COLORS['amber'], lw=1.5,
           label=f'Equal weight = {1/len(feature_cols):.3f}')
ax.legend()

for i, v in enumerate(mean_alpha):
    ax.text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_attention_weights')
plt.show()

In [ ]:
# ── Attention heatmap: weights across samples ────────────────────
# Show attention weights for a window of 60 test-set days
_, alpha_test = attn_model.predict(X_test_s)
window = alpha_test[:60]

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(window.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_yticks(range(len(feature_cols)))
ax.set_yticklabels(feature_cols)
ax.set_xlabel('Test Day Index')
ax.set_ylabel('Feature')
ax.set_title('Temporal Attention Heatmap (First 60 Test Days)', fontweight='bold')
plt.colorbar(im, ax=ax, label='Attention Weight', shrink=0.8)
fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_attention_heatmap')
plt.show()

## 4. Comparison with Linear Model

We compare three approaches:

| Model | Description |
|-------|-------------|
| **Equal-weight baseline** | Prediction = simple average of all features (no learning) |
| **Linear Regression** | sklearn `LinearRegression` on the same features |
| **Temporal Attention** | Our NumPy attention model with learned weights |

If the attention mechanism is useful, it should outperform equal weighting by learning to focus on the most informative features.

In [ ]:
# ── 1. Equal-weight baseline ─────────────────────────────────────
y_eq_test = X_test_s.mean(axis=1)  # simple average of standardised features

# ── 2. Linear Regression ────────────────────────────────────────
lr_model = LinearRegression()
lr_model.fit(X_train_s, y_train)
y_lr_test = lr_model.predict(X_test_s)

# ── 3. Attention model ──────────────────────────────────────────
y_attn_test, _ = attn_model.predict(X_test_s)

# ── Metrics ──────────────────────────────────────────────────────
def eval_model(name, y_true, y_pred):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    # Directional accuracy: did we get the sign right?
    direction = np.mean(np.sign(y_pred) == np.sign(y_true))
    return {'Model': name, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'Dir. Acc.': direction}

results = pd.DataFrame([
    eval_model('Equal Weight',       y_test, y_eq_test),
    eval_model('Linear Regression',  y_test, y_lr_test),
    eval_model('Temporal Attention',  y_test, y_attn_test),
])
results = results.set_index('Model')
print('Out-of-sample performance (test set):')
print(results.to_string(float_format='{:.8f}'.format))
print()

# Highlight the winner
best_mse = results['MSE'].idxmin()
best_dir = results['Dir. Acc.'].idxmax()
print(f'Lowest MSE     : {best_mse}')
print(f'Best direction : {best_dir}')

In [ ]:
# ── Bar chart of metrics ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
model_colors = [COLORS['amber'], COLORS['blue'], COLORS['red']]
model_names  = results.index.tolist()

for ax, metric in zip(axes, ['RMSE', 'MAE', 'Dir. Acc.']):
    vals = results[metric].values
    bars = ax.bar(model_names, vals, color=model_colors, edgecolor='k', linewidth=0.6)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.0005 * (1 if metric != 'Dir. Acc.' else 1),
                f'{v:.5f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_ylabel(metric)

fig.suptitle('Model Comparison — Out-of-Sample', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_comparison')
plt.show()

## 5. Visualization: Actual vs Predicted Returns

In [ ]:
# ── Actual vs Predicted (test set) ───────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Actual returns
axes[0].plot(dates_test, y_test, color='k', lw=0.6, alpha=0.7)
axes[0].set_ylabel('Return')
axes[0].set_title('Actual Returns (Test Set)', fontweight='bold')
axes[0].axhline(0, ls='-', color='gray', lw=0.5)

# Panel 2: Linear Regression predictions vs actual
axes[1].plot(dates_test, y_test, color='k', lw=0.5, alpha=0.4, label='Actual')
axes[1].plot(dates_test, y_lr_test, color=COLORS['blue'], lw=0.8, alpha=0.8,
             label='Linear Regression')
axes[1].set_ylabel('Return')
axes[1].set_title('Linear Regression Predictions', fontweight='bold')
axes[1].legend()
axes[1].axhline(0, ls='-', color='gray', lw=0.5)

# Panel 3: Attention predictions vs actual
axes[2].plot(dates_test, y_test, color='k', lw=0.5, alpha=0.4, label='Actual')
axes[2].plot(dates_test, y_attn_test, color=COLORS['red'], lw=0.8, alpha=0.8,
             label='Temporal Attention')
axes[2].set_ylabel('Return')
axes[2].set_xlabel('Date')
axes[2].set_title('Temporal Attention Predictions', fontweight='bold')
axes[2].legend()
axes[2].axhline(0, ls='-', color='gray', lw=0.5)

fig.suptitle('S&P 500 Return Forecasting — Out-of-Sample', fontsize=14,
             fontweight='bold', y=1.01)
fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_predictions')
plt.show()

In [ ]:
# ── Cumulative return comparison ──────────────────────────────────
# If the model predicts positive return -> go long, else -> go short (or flat)
signal_lr   = np.sign(y_lr_test)
signal_attn = np.sign(y_attn_test)

cum_actual = np.cumsum(y_test)
cum_lr     = np.cumsum(signal_lr * y_test)
cum_attn   = np.cumsum(signal_attn * y_test)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(dates_test, cum_actual, color='k', lw=1.5, alpha=0.6, label='Buy & Hold')
ax.plot(dates_test, cum_lr, color=COLORS['blue'], lw=1.5, label='Linear Regression Signal')
ax.plot(dates_test, cum_attn, color=COLORS['red'], lw=1.5, label='Temporal Attention Signal')
ax.axhline(0, ls='-', color='gray', lw=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Log Return')
ax.set_title('Cumulative Returns: Model-Based Trading Signals', fontweight='bold')
ax.legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_cumulative')
plt.show()

In [ ]:
# ── Attention weights over time (full test set) ─────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Use rolling mean of attention weights for readability
window_size = 20
for i, feat in enumerate(feature_cols):
    color = plt.cm.tab10(i / len(feature_cols))
    rolling_alpha = pd.Series(alpha_test[:, i], index=dates_test).rolling(window_size).mean()
    ax.plot(dates_test, rolling_alpha, lw=1.2, label=feat, color=color)

ax.set_xlabel('Date')
ax.set_ylabel('Attention Weight (20-day MA)')
ax.set_title('Time-Varying Attention Weights Across Features', fontweight='bold')
ax.legend(ncol=4, fontsize=8)

fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_attention_time')
plt.show()

In [ ]:
# ── Training loss curve ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(attn_losses, color=COLORS['red'], lw=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Temporal Attention Model — Training Loss', fontweight='bold')
fig.tight_layout()
save_fig(fig, 'bishop_ch12_tft_loss')
plt.show()

## 6. Key Takeaways

**How transformers improve over simple sequential models for financial time series:**

1. **Adaptive feature weighting.** Unlike a linear model that assigns fixed coefficients, the attention mechanism produces *input-dependent* weights. When volatility spikes, the model can shift attention toward the volatility feature; in calm markets, it can focus on lagged returns.

2. **Variable selection.** The TFT paper (Lim et al., 2021) includes an explicit *variable selection network* that uses attention to gate which inputs matter. Our simplified version demonstrates the same principle: the learned attention weights act as a soft feature selector.

3. **No fixed lag structure.** Traditional AR/ARIMA models impose a fixed lag order. Attention lets the model "look back" flexibly --- attending strongly to $r_{t-1}$ on some days and $r_{t-5}$ on others, depending on the current market context.

4. **Interpretability.** A major advantage of the TFT over black-box deep learning: the attention weights are directly interpretable. We can see *which features the model relied on* for each prediction (cf. the heatmap above).

5. **Scalability to richer inputs.** In a full TFT, the same attention mechanism handles static covariates (sector, country), known future inputs (calendar features), and observed past inputs --- all within a single architecture. Our NumPy demo is the conceptual core of that system.

**Limitations of this demo:**
- We used a single-head, single-layer attention for clarity. Production TFTs use multi-head attention across multiple temporal horizons.
- Financial returns are notoriously hard to predict; even small improvements in MSE or directional accuracy can be economically meaningful.
- This is a **teaching notebook**, not investment advice.

**Reference:** Bishop & Bishop (2024), *Deep Learning: Foundations and Concepts*, Chapter 12 --- Transformers.

In [ ]:
takeaways = [
    '1. Attention computes input-dependent weights -- unlike fixed linear coefficients.',
    '2. The attention heatmap shows which features the model relies on for each prediction.',
    '3. Temporal attention adapts: volatility features gain weight during turbulent periods.',
    '4. Even a simple 1-layer attention can match or beat OLS on noisy financial data.',
    '5. The full TFT extends this with multi-head attention, gating, and quantile outputs.',
]
print('Key Takeaways:')
for t in takeaways:
    print(t)